# Phase 3 — Filtered Dataset Builder

Pipeline này biến raw data thành dataset sạch, sẵn sàng cho LightFM.

**Ba giai đoạn chính:**

| Giai đoạn | Tên | Input | Output |
|-----------|-----|-------|--------|
| 3A | **K-core Filtering** (k=8) | raw interactions JSON | filtered interactions JSON |
| 3B | **Genre Cleaning** | raw books JSON | cleaned genre list + removed tags log |
| 3C | **Genre Transform** | cleaned genres | normalized top-k genre features |

> Chạy tuần tự từ trên xuống. Mỗi giai đoạn phụ thuộc vào kết quả của giai đoạn trước.

## Cell 1 — Config & Imports

Tất cả tham số quan trọng được tập trung tại một chỗ để dễ điều chỉnh.  
Thay đổi `K_CORE`, `TOP_K_GENRES`, hoặc đường dẫn file ở đây — không cần sửa code bên dưới.

In [ ]:
from pathlib import Path
from collections import Counter
import json
import re
import unicodedata
from datetime import datetime

import numpy as np
import pandas as pd

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR          = Path.cwd()
INTERACTIONS_PATH = DATA_DIR / "goodreads_interactions_children.json"
BOOKS_PATH        = DATA_DIR / "goodreads_books_children.json"

OUTPUT_DIR        = DATA_DIR / "phase3_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── K-core tham số (từ kết quả EDA Phase 2) ──────────────────────────────────
K_CORE = 8                   # min interactions cho cả user lẫn item

# ── Genre tham số ─────────────────────────────────────────────────────────────
TOP_K_GENRES       = 5       # số genres giữ lại cho mỗi cuốn sách
MIN_GENRE_DOC_FREQ = 10      # genre phải xuất hiện ở ít nhất N cuốn sách mới hợp lệ

# ── Streaming config ──────────────────────────────────────────────────────────
DEDUP_KEYS             = ("user_id", "book_id")
MAX_INTERACTION_LINES  = None   # None = đọc hết file
MAX_BOOK_LINES         = None

# ── Date filter (từ EDA: chỉ giữ interactions từ 2007 trở đi) ─────────────────
DATE_FORMAT           = "%a %b %d %H:%M:%S %z %Y"
APPLY_YEAR_FILTER     = True
MIN_INTERACTION_YEAR  = 2007

print("✅ Config loaded.")
print(f"   K_CORE           = {K_CORE}")
print(f"   TOP_K_GENRES     = {TOP_K_GENRES}")
print(f"   MIN_GENRE_DOC_FREQ = {MIN_GENRE_DOC_FREQ}")
print(f"   INTERACTIONS     : {INTERACTIONS_PATH.name}")
print(f"   BOOKS            : {BOOKS_PATH.name}")
print(f"   OUTPUT_DIR       : {OUTPUT_DIR}")

---
## Giai đoạn 3A — K-core Filtering (k=8)

### Tại sao cần lặp lại?

K-core không thể thực hiện chỉ trong một lần quét vì có hiệu ứng **cascade**:
khi một user bị loại, các sách mà user đó từng tương tác mất đi 1 lượt —
và cuốn sách đó có thể bây giờ không còn đủ k interactions nữa, dẫn đến bị loại theo.
Khi cuốn sách bị loại, lại ảnh hưởng đến các user khác… cứ như vậy.

Thuật toán lặp cho đến khi không còn node nào cần loại thêm (**hội tụ**).

### Ba bước trong giai đoạn này:

**Bước 1** — Streaming Pass đầu tiên: tính degree của từng user và item trong chế độ `date_added`
(implicit mode, chỉ giữ records có date_added hợp lệ và năm ≥ 2007).

**Bước 2** — Lặp k-core cho đến hội tụ: ở mỗi vòng lặp, loại bỏ tất cả
user có degree < k và item có degree < k, sau đó tính lại degree trên tập còn lại.

**Bước 3** — Streaming Pass cuối: ghi ra file JSON mới chỉ chứa
interactions của (user, item) đã vượt qua k-core.

### Cell 3A-1 — Streaming Pass: Tính degree ban đầu

Đọc toàn bộ file interactions một lần để xây dựng `user_degree` và `book_degree`.
Đây là "ảnh chụp" ban đầu của mạng lưới trước khi lọc.

In [ ]:
def parse_year_from_date(date_str: str) -> int:
    """Parse year từ date_added string. Trả về -1 nếu lỗi."""
    try:
        dt = pd.to_datetime(date_str, format=DATE_FORMAT, errors="coerce")
        return int(dt.year) if not pd.isna(dt) else -1
    except Exception:
        return -1


def is_valid_interaction(rec: dict) -> bool:
    """
    Kiểm tra record có hợp lệ theo chế độ 'date_added' không.
    Điều kiện: phải có date_added và năm >= MIN_INTERACTION_YEAR.
    """
    date_raw = str(rec.get("date_added", "") or "").strip()
    if not date_raw:
        return False
    yr = parse_year_from_date(date_raw)
    if yr == -1:
        return False
    if APPLY_YEAR_FILTER and yr < MIN_INTERACTION_YEAR:
        return False
    return True


# ── Pass 1: đếm degree ────────────────────────────────────────────────────────
user_degree = Counter()
book_degree = Counter()

rows_raw       = 0
rows_deduped   = 0
rows_invalid   = 0
seen_keys      = set()

print("🔄 Pass 1: Đang đọc interactions để tính degree...")

with INTERACTIONS_PATH.open("r", encoding="utf-8") as f:
    for line_no, raw_line in enumerate(f, 1):
        if MAX_INTERACTION_LINES and line_no > MAX_INTERACTION_LINES:
            break

        line = raw_line.strip()
        if not line:
            continue

        try:
            rec = json.loads(line)
        except json.JSONDecodeError:
            continue

        uid = str(rec.get("user_id", "") or "").strip()
        bid = str(rec.get("book_id", "") or "").strip()
        if not uid or not bid:
            continue

        rows_raw += 1

        # Dedup theo (user_id, book_id)
        key = (uid, bid)
        if key in seen_keys:
            continue
        seen_keys.add(key)

        # Kiểm tra điều kiện date_added
        if not is_valid_interaction(rec):
            rows_invalid += 1
            continue

        rows_deduped += 1
        user_degree[uid] += 1
        book_degree[bid] += 1

print(f"✅ Pass 1 hoàn thành.")
print(f"   Rows raw          : {rows_raw:,}")
print(f"   Rows sau dedup    : {rows_deduped + rows_invalid:,}")
print(f"   Rows invalid date : {rows_invalid:,}")
print(f"   Rows hợp lệ       : {rows_deduped:,}")
print(f"   Unique users      : {len(user_degree):,}")
print(f"   Unique items      : {len(book_degree):,}")

### Cell 3A-2 — K-core Iterative: Lọc đến hội tụ

Vòng lặp này hoạt động hoàn toàn **in-memory** — không cần đọc file lại.
Ở mỗi iteration, ta xây dựng lại degree từ các node còn sống,
tìm node vi phạm điều kiện k, loại chúng ra, rồi lặp lại.
Khi không còn node nào bị loại → **converged**.

In [ ]:
# ── K-core iterative filtering ────────────────────────────────────────────────
# Bắt đầu với toàn bộ set user/item từ Pass 1
active_users = set(user_degree.keys())
active_items = set(book_degree.keys())

# Ta cần rebuild degree từ raw interactions mỗi vòng — nhưng để
# tránh đọc file nhiều lần, ta cache toàn bộ valid interactions vào RAM.
# Với dataset vài triệu records, list of tuples vẫn fit trong memory.

print("💾 Cache valid interactions vào memory...")
valid_interactions = []  # list of (user_id, book_id)

seen_cache = set()
with INTERACTIONS_PATH.open("r", encoding="utf-8") as f:
    for raw_line in f:
        line = raw_line.strip()
        if not line:
            continue
        try:
            rec = json.loads(line)
        except json.JSONDecodeError:
            continue

        uid = str(rec.get("user_id", "") or "").strip()
        bid = str(rec.get("book_id", "") or "").strip()
        if not uid or not bid:
            continue

        key = (uid, bid)
        if key in seen_cache:
            continue
        seen_cache.add(key)

        if is_valid_interaction(rec):
            valid_interactions.append((uid, bid))

print(f"   Cached {len(valid_interactions):,} valid interactions.")

# ── Vòng lặp k-core ──────────────────────────────────────────────────────────
iteration_log = []

print(f"\n🔄 Bắt đầu k-core với k={K_CORE}...")

for iteration in range(1, 101):   # max 100 iterations, thực tế thường hội tụ < 10
    # Tính lại degree chỉ trên active nodes
    cur_user_deg = Counter()
    cur_item_deg = Counter()

    for uid, bid in valid_interactions:
        if uid in active_users and bid in active_items:
            cur_user_deg[uid] += 1
            cur_item_deg[bid] += 1

    # Tìm nodes vi phạm điều kiện
    low_users = {u for u, d in cur_user_deg.items() if d < K_CORE}
    low_items = {i for i, d in cur_item_deg.items() if d < K_CORE}

    # Cũng loại những node không xuất hiện trong iteration này
    # (có thể đã bị cô lập hoàn toàn sau các vòng trước)
    vanished_users = active_users - set(cur_user_deg.keys())
    vanished_items = active_items - set(cur_item_deg.keys())
    low_users |= vanished_users
    low_items |= vanished_items

    log_entry = {
        "iteration"     : iteration,
        "active_users"  : len(active_users),
        "active_items"  : len(active_items),
        "active_edges"  : sum(cur_user_deg.values()),
        "removed_users" : len(low_users),
        "removed_items" : len(low_items),
    }
    iteration_log.append(log_entry)

    print(f"   Iter {iteration:2d}: users={len(active_users):,}  items={len(active_items):,}  "
          f"edges={log_entry['active_edges']:,}  "
          f"removed_u={len(low_users):,}  removed_i={len(low_items):,}")

    # Nếu không còn gì để loại → đã hội tụ
    if not low_users and not low_items:
        print(f"\n✅ Hội tụ sau {iteration} iterations.")
        break

    active_users -= low_users
    active_items -= low_items

# Lưu final degree để dùng sau
final_user_deg = cur_user_deg
final_item_deg = cur_item_deg
final_edges    = sum(final_user_deg.values())

# Tính density
density_after = final_edges / (len(active_users) * len(active_items)) if active_users and active_items else 0

print(f"\n── Kết quả sau k-core ───────────────────────────────")
print(f"   Users  : {len(active_users):,}  (degree min={min(final_user_deg.values()):,}  max={max(final_user_deg.values()):,})")
print(f"   Items  : {len(active_items):,}  (degree min={min(final_item_deg.values()):,}  max={max(final_item_deg.values()):,})")
print(f"   Edges  : {final_edges:,}")
print(f"   Density: {density_after:.6f}")

# Hiển thị iteration log
pd.set_option("display.max_rows", 20)
display(pd.DataFrame(iteration_log))

### Cell 3A-3 — Export: Ghi filtered interactions ra file JSON

Đọc file gốc lần cuối, chỉ ghi ra những records mà cả `user_id` lẫn `book_id`
đều nằm trong `active_users` và `active_items` (đã qua k-core).

File output giữ nguyên **toàn bộ fields** của record gốc — không bỏ thêm gì cả —
để Phase 4 có thể dùng thêm thông tin nếu cần (ví dụ: `rating`, `date_added`).

In [ ]:
# ── Export filtered interactions ──────────────────────────────────────────────
FILTERED_INTERACTIONS_PATH = OUTPUT_DIR / f"interactions_kcore{K_CORE}.json"

kept_rows         = 0
skipped_not_kcore = 0
skipped_invalid   = 0
skipped_dup       = 0
seen_export       = set()

print(f"💾 Đang ghi filtered interactions → {FILTERED_INTERACTIONS_PATH.name}...")

with INTERACTIONS_PATH.open("r", encoding="utf-8") as fin, \
     FILTERED_INTERACTIONS_PATH.open("w", encoding="utf-8") as fout:

    for raw_line in fin:
        line = raw_line.strip()
        if not line:
            continue
        try:
            rec = json.loads(line)
        except json.JSONDecodeError:
            continue

        uid = str(rec.get("user_id", "") or "").strip()
        bid = str(rec.get("book_id", "") or "").strip()
        if not uid or not bid:
            continue

        # Dedup
        key = (uid, bid)
        if key in seen_export:
            skipped_dup += 1
            continue
        seen_export.add(key)

        # Kiểm tra date_added validity
        if not is_valid_interaction(rec):
            skipped_invalid += 1
            continue

        # Kiểm tra k-core membership
        if uid not in active_users or bid not in active_items:
            skipped_not_kcore += 1
            continue

        fout.write(json.dumps(rec, ensure_ascii=False) + "\n")
        kept_rows += 1

print(f"\n✅ Export hoàn thành.")
print(f"   Kept rows         : {kept_rows:,}")
print(f"   Skipped (not kcore): {skipped_not_kcore:,}")
print(f"   Skipped (invalid) : {skipped_invalid:,}")
print(f"   Skipped (dup)     : {skipped_dup:,}")
print(f"   File size         : {FILTERED_INTERACTIONS_PATH.stat().st_size / 1024**2:.1f} MB")

# Lưu iteration log ra CSV để tham khảo
kcore_log_df = pd.DataFrame(iteration_log)
kcore_log_df.to_csv(OUTPUT_DIR / "kcore_iteration_log.csv", index=False, encoding="utf-8-sig")
print(f"   Iteration log     : {OUTPUT_DIR / 'kcore_iteration_log.csv'}")

---
## Giai đoạn 3B — Genre Cleaning

### Vấn đề cần giải quyết

Goodreads lưu thể loại sách trong trường `popular_shelves` — một danh sách
các shelf mà người dùng tự đặt tên. Vì không có kiểm soát từ phía Goodreads,
danh sách này chứa lẫn lộn ba loại thông tin hoàn toàn khác nhau:

- **Genre thực sự** (cần giữ): `fantasy`, `children`, `picture-books`, `adventure`
- **Shelf trạng thái** (cần loại): `to-read`, `currently-reading`, `did-not-finish`
- **Shelf cá nhân / format** (cần loại): `kindle`, `owned`, `my-library`, `5-stars`

### Hai chiến lược kết hợp

**Chiến lược 1 — Blacklist:** Xây dựng tập hợp các tên shelf "biết chắc là rác".
Phủ được khoảng 80-90% noise thường gặp. Nhanh và chính xác với những case rõ ràng.

**Chiến lược 2 — Heuristic:** Các quy tắc tổng quát để bắt những tên shelf lạ
mà blacklist chưa cover. Ví dụ: tên quá ngắn, chứa số, bắt đầu bằng "my-", v.v.

Mọi shelf bị loại đều được **ghi vào log riêng** để bạn có thể review
và phát hiện nếu có genre hợp lệ bị loại nhầm.

### Cell 3B-1 — Định nghĩa Blacklist & Heuristic Rules

Đây là cell quan trọng nhất của giai đoạn 3B. Hãy đọc từng nhóm trong blacklist
để hiểu tại sao chúng cần bị loại.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CHIẾN LƯỢC 1 — BLACKLIST
# Mỗi nhóm có comment giải thích lý do loại bỏ
# ══════════════════════════════════════════════════════════════════════════════

SHELF_BLACKLIST = {

    # ── Nhóm 1: Trạng thái đọc ───────────────────────────────────────────────
    # Đây là thông tin về HÀNH VI của user, không phải NỘI DUNG của sách.
    # Nếu đưa vào model, tất cả sách đều sẽ có genre "to-read" → vô nghĩa.
    "to-read", "currently-reading", "read", "reading",
    "did-not-finish", "dnf", "re-read", "reread",
    "abandoned", "unfinished", "stopped-reading",
    "maybe", "maybe-read", "want-to-read",
    "have-read", "already-read", "finished",

    # ── Nhóm 2: Sở hữu / vị trí vật lý ──────────────────────────────────────
    # Thông tin về việc user SỞ HỮU sách như thế nào, không liên quan đến genre.
    "owned", "own", "own-it", "books-i-own", "owned-books",
    "my-books", "my-library", "my-collection", "my-shelf",
    "home-library", "bookshelf", "bookcase",
    "wish-list", "wishlist", "want", "want-to-buy", "to-buy",
    "borrowed", "library-book", "library", "library-loan",

    # ── Nhóm 3: Format / phương tiện đọc ─────────────────────────────────────
    # Mô tả định dạng vật lý, không phải nội dung thể loại.
    "ebook", "e-book", "kindle", "kindle-books",
    "audiobook", "audio", "audio-books", "audible",
    "paperback", "hardcover", "hardback",
    "digital", "pdf",

    # ── Nhóm 4: Danh sách / thách thức đọc ───────────────────────────────────
    # Reading challenges hay lists cá nhân, không phải genre.
    "favorites", "favourite", "favourite-books",
    "best", "best-books", "best-of", "top-books",
    "default", "general", "misc", "miscellaneous", "other",
    "starred", "five-stars", "4-stars", "3-stars",
    "to-reread", "series", "series-to-read",
    "book-club", "bookclub", "read-aloud",
    "challenge", "reading-challenge", "2023-reads", "2022-reads",
    "2021-reads", "2020-reads", "2019-reads", "2018-reads",
    "100-books", "reading-list",

    # ── Nhóm 5: Ngôn ngữ ─────────────────────────────────────────────────────
    # Ngôn ngữ không phải genre sách.
    "english", "spanish", "french", "german", "italian",
    "portuguese", "dutch", "swedish", "norwegian",
    "japanese", "chinese", "korean", "arabic",
    "translated", "translation",

    # ── Nhóm 6: Thông tin tác giả / series ───────────────────────────────────
    # Tên tác giả hoặc series không phải genre (chúng là feature riêng).
    "author", "authors", "same-author",
    "sequels", "series-books", "standalone",
}

# ══════════════════════════════════════════════════════════════════════════════
# CHIẾN LƯỢC 2 — HEURISTIC RULES
# Các hàm kiểm tra để bắt những pattern lạ mà blacklist không cover
# ══════════════════════════════════════════════════════════════════════════════

def heuristic_is_junk(name: str) -> tuple[bool, str]:
    """
    Kiểm tra một tên shelf có phải 'rác' theo heuristic không.
    Trả về (is_junk: bool, reason: str).
    """
    # Rule 1: Quá ngắn — tên có ≤ 2 ký tự thường là label tùy tiện
    if len(name) <= 2:
        return True, "too_short"

    # Rule 2: Chứa toàn số hoặc pattern số-năm
    # Ví dụ: "2023", "100", "1001-books"
    if re.fullmatch(r"\d+", name):
        return True, "pure_number"

    # Rule 3: Pattern "số-unit" như "5-stars", "4-star", "100-books"
    if re.match(r"^\d+[-_](stars?|books?|reads?|pages?)$", name):
        return True, "rating_or_count_pattern"

    # Rule 4: Bắt đầu bằng "my-" hay "i-" → ghi chú cá nhân
    # Ví dụ: "my-favorites", "i-own", "my-2023"
    if name.startswith(("my-", "my_", "i-own", "i-have")):
        return True, "personal_prefix"

    # Rule 5: Chứa năm 4 chữ số (reading year lists)
    # Ví dụ: "read-2022", "books-2023", "2021-challenge"
    if re.search(r"(19|20)\d{2}", name):
        return True, "contains_year"

    # Rule 6: Tên là email-style hoặc URL-like
    if "@" in name or "http" in name:
        return True, "url_or_email"

    # Rule 7: Quá dài — tên genre hợp lệ thường không quá 40 ký tự
    if len(name) > 40:
        return True, "too_long"

    return False, ""


print("✅ Blacklist và Heuristic rules đã được định nghĩa.")
print(f"   Blacklist size: {len(SHELF_BLACKLIST)} entries")
print(f"   Heuristic rules: 7 rules")

### Cell 3B-2 — Streaming Pass: Đọc books.json và clean genres

Đọc file books, với mỗi cuốn sách thực hiện:

1. Lấy `popular_shelves` (list các dict `{name, count}`)
2. Với mỗi shelf: kiểm tra blacklist trước, rồi heuristic
3. Ghi lại những shelf bị loại vào log để review sau
4. Lưu genres sạch vào dict `book_clean_genres`

In [ ]:
# ── Storage structures ────────────────────────────────────────────────────────
book_clean_genres  = {}   # {book_id: [genre_name, ...]} — genres đã clean
removed_tags_log   = []   # log mỗi shelf bị loại: {book_id, shelf, reason}

# Counter để thống kê
genre_kept_counter   = Counter()   # genre nào được giữ → xuất hiện bao nhiêu cuốn
genre_removed_by_reason = Counter()  # lý do loại → số lần

book_rows       = 0
book_no_shelves = 0

# Set book_ids đã qua k-core để chỉ process sách liên quan
# (tùy chọn — bỏ dòng này nếu muốn clean toàn bộ books file)
kcore_book_ids = active_items   # set từ giai đoạn 3A

print(f"🔄 Đang đọc books và clean genres...")
print(f"   Chỉ process {len(kcore_book_ids):,} books đã qua k-core.")

with BOOKS_PATH.open("r", encoding="utf-8") as f:
    for line_no, raw_line in enumerate(f, 1):
        if MAX_BOOK_LINES and line_no > MAX_BOOK_LINES:
            break

        line = raw_line.strip()
        if not line:
            continue
        try:
            rec = json.loads(line)
        except json.JSONDecodeError:
            continue

        book_id = str(rec.get("book_id", "") or "").strip()
        if not book_id:
            continue

        # Chỉ xử lý sách trong k-core (tùy chọn — comment dòng dưới nếu muốn all books)
        if book_id not in kcore_book_ids:
            continue

        book_rows += 1

        # ── Lấy popular_shelves ───────────────────────────────────────────────
        shelves_raw = rec.get("popular_shelves", []) or []
        if not shelves_raw:
            book_no_shelves += 1
            book_clean_genres[book_id] = []
            continue

        kept_genres   = []
        book_removed  = []

        for shelf in shelves_raw:
            # Lấy tên shelf — có thể là dict {"name": ..., "count": ...} hoặc string
            if isinstance(shelf, dict):
                shelf_name = str(shelf.get("name", "") or "").strip().lower()
            else:
                shelf_name = str(shelf).strip().lower()

            if not shelf_name:
                continue

            # ── CHIẾN LƯỢC 1: Kiểm tra Blacklist ─────────────────────────────
            if shelf_name in SHELF_BLACKLIST:
                book_removed.append({"shelf": shelf_name, "reason": "blacklist"})
                genre_removed_by_reason["blacklist"] += 1
                continue

            # ── CHIẾN LƯỢC 2: Kiểm tra Heuristic ─────────────────────────────
            is_junk, reason = heuristic_is_junk(shelf_name)
            if is_junk:
                book_removed.append({"shelf": shelf_name, "reason": f"heuristic:{reason}"})
                genre_removed_by_reason[f"heuristic:{reason}"] += 1
                continue

            # ── Passed cả hai chiến lược → giữ lại ───────────────────────────
            kept_genres.append(shelf_name)
            genre_kept_counter[shelf_name] += 1

        # Ghi log những shelf bị remove của cuốn sách này
        for entry in book_removed:
            removed_tags_log.append({
                "book_id": book_id,
                "removed_shelf": entry["shelf"],
                "reason": entry["reason"],
            })

        book_clean_genres[book_id] = kept_genres

print(f"\n✅ Genre cleaning hoàn thành.")
print(f"   Books processed    : {book_rows:,}")
print(f"   Books no shelves   : {book_no_shelves:,}")
print(f"   Unique books với genres: {sum(1 for g in book_clean_genres.values() if g):,}")
print(f"   Total removed tags : {len(removed_tags_log):,}")
print(f"\n── Removed by reason ────────────────────────────────")
for reason, count in genre_removed_by_reason.most_common():
    print(f"   {reason:<30} : {count:,}")

### Cell 3B-3 — Review: Kiểm tra Tags bị loại có nhầm không?

Đây là bước **quality control** quan trọng. Mục đích là phát hiện những genre
hợp lệ bị loại nhầm bởi blacklist hoặc heuristic.

Hãy nhìn vào cột `removed_shelf` và tự hỏi: *"Cái này có thực sự mô tả
nội dung sách không?"* Nếu có, hãy xóa nó khỏi `SHELF_BLACKLIST` hoặc
điều chỉnh heuristic rule tương ứng, rồi chạy lại Cell 3B-2.

In [ ]:
# ── Export removed tags log để review ────────────────────────────────────────
removed_log_df = pd.DataFrame(removed_tags_log)
REMOVED_TAGS_PATH = OUTPUT_DIR / "removed_shelf_tags_log.csv"
removed_log_df.to_csv(REMOVED_TAGS_PATH, index=False, encoding="utf-8-sig")
print(f"💾 Removed tags log: {REMOVED_TAGS_PATH}  ({len(removed_log_df):,} entries)")

# ── Nhìn vào những shelf bị loại theo heuristic (dễ nhầm nhất) ───────────────
print("\n── Top 50 shelves bị loại bởi HEURISTIC (cần review kỹ nhất) ──")
heuristic_removed = (
    removed_log_df[removed_log_df["reason"].str.startswith("heuristic")]
    .groupby("removed_shelf")
    .agg(
        times_removed=("removed_shelf", "count"),
        reasons=("reason", lambda x: ", ".join(sorted(set(x))))
    )
    .sort_values("times_removed", ascending=False)
    .head(50)
    .reset_index()
)
display(heuristic_removed)

# ── Top shelves bị loại bởi blacklist (để verify blacklist đúng) ──────────────
print("\n── Top 30 shelves bị loại bởi BLACKLIST ──")
blacklist_removed = (
    removed_log_df[removed_log_df["reason"] == "blacklist"]
    .groupby("removed_shelf")
    .size()
    .sort_values(ascending=False)
    .head(30)
    .to_frame("times_removed")
    .reset_index()
    .rename(columns={"removed_shelf": "shelf"})
)
display(blacklist_removed)

# ── Tổng hợp nhanh ────────────────────────────────────────────────────────────
print("\n── Kept genres (top 30 phổ biến nhất sau clean) ──")
kept_top30 = (
    pd.Series(genre_kept_counter)
    .sort_values(ascending=False)
    .head(30)
    .to_frame("book_count")
    .reset_index()
    .rename(columns={"index": "genre"})
)
display(kept_top30)

print("\n💡 Nếu thấy genre hợp lệ trong 'heuristic_removed', hãy:")
print("   1. Xóa nó khỏi SHELF_BLACKLIST (nếu nó bị blacklist)")
print("   2. Hoặc thêm nó vào exceptions trong heuristic_is_junk()")
print("   3. Rồi chạy lại Cell 3B-2")

---
## Giai đoạn 3C — Genre Transform

### Hai việc cần làm

Sau khi đã loại bỏ shelf rác, genres còn lại vẫn cần được **chuẩn hóa**
trước khi đưa vào LightFM. Lý do là cùng một thể loại có thể được viết
theo nhiều cách khác nhau: `picture-books`, `picture_books`, `PictureBooks`,
`picture books` — chúng phải trở thành một token duy nhất để được map
vào cùng một embedding vector trong model.

**Bước 3C-1 — Normalize tên genre:** chuẩn hóa chuỗi ký tự để hợp nhất
các biến thể khác nhau của cùng một genre.

**Bước 3C-2 — Xây dựng global vocabulary và chọn top-k:** đếm tần suất
của từng genre trên toàn bộ dataset, lấy top-k phổ biến nhất.
Chỉ những genre nằm trong top-k vocabulary mới được giữ lại trong
feature matrix cuối cùng.

**Bước 3C-3 — Build final genre mapping:** với mỗi cuốn sách, lấy
danh sách genres đã normalize và filter chỉ giữ những genre trong vocabulary.

### Cell 3C-1 — Normalize Genre Names

Hàm normalize thực hiện bốn thao tác theo thứ tự cố định.
Thứ tự quan trọng: phải unicode-normalize trước, rồi mới lowercase,
vì một số ký tự unicode có thể thay đổi sau khi lowercase.

In [ ]:
def normalize_genre(name: str) -> str:
    """
    Chuẩn hóa tên genre thành dạng chuẩn để so sánh và đếm.

    Pipeline:
    1. Unicode normalization (NFKD) → decompose ký tự đặc biệt
       Ví dụ: 'café' → 'cafe', 'naïve' → 'naive'
    2. ASCII encode/decode → loại bỏ ký tự non-ASCII
    3. Lowercase → đồng nhất hoa/thường
    4. Thay thế separator (-, _, .) bằng space
    5. Loại bỏ ký tự không phải chữ/số/space
    6. Collapse whitespace
    7. Strip đầu cuối
    """
    if not name:
        return ""

    # Bước 1-2: Xử lý unicode — đưa về ASCII thuần
    name = unicodedata.normalize("NFKD", name)
    name = name.encode("ascii", errors="ignore").decode("ascii")

    # Bước 3: Lowercase
    name = name.lower()

    # Bước 4: Thay separator bằng space để "picture-books" = "picture books"
    name = re.sub(r"[-_./\\]", " ", name)

    # Bước 5: Chỉ giữ chữ cái, số, và space
    name = re.sub(r"[^a-z0-9 ]", "", name)

    # Bước 6-7: Collapse multiple spaces và strip
    name = re.sub(r"\s+", " ", name).strip()

    return name


# ── Test normalize function ───────────────────────────────────────────────────
test_cases = [
    ("picture-books",   "picture books"),
    ("Sci-Fi",          "sci fi"),
    ("YOUNG_ADULT",     "young adult"),
    ("café-books",      "caf books"),    # ký tự é bị drop sau ASCII encode
    ("fantasy",         "fantasy"),
    ("5-stars",         "5 stars"),      # sẽ bị heuristic bắt trước khi đến đây
    ("my.shelf",        "my shelf"),
    ("Science-Fiction", "science fiction"),
]

print("── Kiểm tra normalize_genre ──────────────────────────")
all_pass = True
for input_val, expected in test_cases:
    result = normalize_genre(input_val)
    status = "✅" if result == expected else "❌"
    if result != expected:
        all_pass = False
    print(f"   {status}  '{input_val}' → '{result}'  (expected: '{expected}')")

print()
if all_pass:
    print("✅ Tất cả test cases passed.")
else:
    print("⚠️  Có test case không khớp — kiểm tra lại normalize_genre().")

### Cell 3C-2 — Apply Normalize + Build Global Vocabulary

Áp dụng `normalize_genre` lên toàn bộ `book_clean_genres`,
sau đó đếm **document frequency** (số sách có genre đó)
để xây dựng global vocabulary và chọn top-k.

Tại sao dùng **document frequency** chứ không phải raw count?
Vì bạn muốn biết genre đó phổ biến ở bao nhiêu **cuốn sách**,
không phải xuất hiện bao nhiêu lần — nếu một sách được gán "fantasy"
bởi 50 user thì vẫn chỉ đóng góp 1 vào document frequency.

In [ ]:
# ── Bước 1: Normalize tất cả genres ──────────────────────────────────────────
book_normalized_genres = {}   # {book_id: [normalized_genre, ...]}

for book_id, genres in book_clean_genres.items():
    normalized = []
    for g in genres:
        norm = normalize_genre(g)
        # Sau khi normalize, kiểm tra lại: nếu tên trở nên quá ngắn thì bỏ
        if norm and len(norm) > 2:
            normalized.append(norm)
    # Deduplicate trong cùng một sách (sau normalize có thể trùng)
    seen_in_book = set()
    deduped = []
    for g in normalized:
        if g not in seen_in_book:
            deduped.append(g)
            seen_in_book.add(g)
    book_normalized_genres[book_id] = deduped

# ── Bước 2: Tính document frequency ──────────────────────────────────────────
# Mỗi cuốn sách chỉ đóng góp 1 cho mỗi genre dù genre đó lặp lại
genre_doc_freq = Counter()
for book_id, genres in book_normalized_genres.items():
    for g in set(genres):   # set() để deduplicate trong cùng một sách
        genre_doc_freq[g] += 1

# ── Bước 3: Thống kê phân phối genres/book ────────────────────────────────────
genres_per_book = [len(g) for g in book_normalized_genres.values()]
genres_per_book_arr = np.array(genres_per_book)

print(f"── Genres sau normalize ─────────────────────────────")
print(f"   Unique genres (toàn dataset)  : {len(genre_doc_freq):,}")
print(f"   Avg genres/book               : {genres_per_book_arr.mean():.2f}")
print(f"   Median genres/book            : {float(np.median(genres_per_book_arr)):.1f}")
print(f"   Books với 0 genre sau clean   : {(genres_per_book_arr == 0).sum():,}")

# ── Bước 4: Lọc theo MIN_GENRE_DOC_FREQ trước khi chọn top-k ─────────────────
# Loại bỏ những genre quá hiếm (chỉ xuất hiện ở rất ít sách)
# vì model không thể học pattern từ chúng
qualified_genres = {
    g: cnt for g, cnt in genre_doc_freq.items()
    if cnt >= MIN_GENRE_DOC_FREQ
}
print(f"\n── Sau khi lọc doc_freq >= {MIN_GENRE_DOC_FREQ} ────────────────────")
print(f"   Genres qualified              : {len(qualified_genres):,}  "
      f"(loại bỏ {len(genre_doc_freq) - len(qualified_genres):,} genres hiếm)")

# ── Bước 5: Chọn top-k genres ─────────────────────────────────────────────────
top_genres_ranked = sorted(qualified_genres.items(), key=lambda x: x[1], reverse=True)
top_k_genre_list  = [g for g, _ in top_genres_ranked[:TOP_K_GENRES]]
top_k_genre_set   = set(top_k_genre_list)   # set để O(1) lookup

print(f"\n── Top-{TOP_K_GENRES} genres được chọn ─────────────────────────")
print(f"   {'Rank':<6} {'Genre':<30} {'Doc Freq':>10}")
print(f"   {'-'*6} {'-'*30} {'-'*10}")
for rank, genre in enumerate(top_k_genre_list, 1):
    freq = genre_doc_freq[genre]
    print(f"   {rank:<6} {genre:<30} {freq:>10,}")

# ── Bước 6: Tính coverage của top-k vocabulary ────────────────────────────────
# Coverage = % sách có ít nhất 1 genre trong top-k vocabulary
books_with_topk = sum(
    1 for genres in book_normalized_genres.values()
    if any(g in top_k_genre_set for g in genres)
)
coverage_pct = 100.0 * books_with_topk / len(book_normalized_genres) if book_normalized_genres else 0

print(f"\n── Coverage Analysis ────────────────────────────────")
print(f"   Books có ít nhất 1 genre trong top-{TOP_K_GENRES}: {books_with_topk:,}  ({coverage_pct:.1f}%)")
print(f"   Books không có genre nào: {len(book_normalized_genres) - books_with_topk:,}")

### Cell 3C-3 — Build Final Genre Mapping

Tạo ra mapping cuối cùng `book_final_genres`:
với mỗi cuốn sách, chỉ giữ lại những genre nằm trong `top_k_genre_set`.
Genres được sắp xếp theo **document frequency giảm dần** để đảm bảo
tính nhất quán — nếu có nhiều sách dùng cùng một genre, genre đó được ưu tiên.

In [ ]:
# ── Build final genre mapping ─────────────────────────────────────────────────
book_final_genres = {}   # {book_id: [genre, ...]} — chỉ chứa top-k genres

for book_id, genres in book_normalized_genres.items():
    # Filter: chỉ giữ genre trong top-k vocabulary
    valid = [g for g in genres if g in top_k_genre_set]

    # Sort theo document frequency giảm dần (genre phổ biến nhất lên đầu)
    valid_sorted = sorted(valid, key=lambda g: genre_doc_freq.get(g, 0), reverse=True)

    book_final_genres[book_id] = valid_sorted

# ── Thống kê kết quả cuối ─────────────────────────────────────────────────────
final_genres_per_book = [len(g) for g in book_final_genres.values()]
final_arr = np.array(final_genres_per_book)

dist_rows = []
for n_genres in range(TOP_K_GENRES + 1):
    count = (final_arr == n_genres).sum()
    pct   = 100.0 * count / len(final_arr) if len(final_arr) else 0
    dist_rows.append({"n_genres": n_genres, "book_count": count, "pct": round(pct, 2)})

print(f"── Phân phối số genres/book sau transform ───────────")
display(pd.DataFrame(dist_rows))

print(f"\n── Tóm tắt ─────────────────────────────────────────")
print(f"   Books với 0 genre (sẽ dùng ID embedding): {(final_arr == 0).sum():,}")
print(f"   Books với đủ {TOP_K_GENRES} genres             : {(final_arr == TOP_K_GENRES).sum():,}")
print(f"   Avg genres/book sau transform    : {final_arr.mean():.2f}")

---
## Export Cuối — Lưu Filtered Books JSON

Ghi ra file `books_filtered.json` chứa metadata của những cuốn sách
đã qua k-core, với trường `genres_clean` chứa kết quả transform.

File này sẽ là input cho Phase 4 (Feature Engineering + LightFM).

In [ ]:
# ── Export filtered books với genres đã clean ─────────────────────────────────
FILTERED_BOOKS_PATH = OUTPUT_DIR / f"books_filtered_kcore{K_CORE}_top{TOP_K_GENRES}genres.json"

kept_books  = 0
total_books = 0

print(f"💾 Đang ghi filtered books → {FILTERED_BOOKS_PATH.name}...")

with BOOKS_PATH.open("r", encoding="utf-8") as fin, \
     FILTERED_BOOKS_PATH.open("w", encoding="utf-8") as fout:

    for raw_line in fin:
        line = raw_line.strip()
        if not line:
            continue
        try:
            rec = json.loads(line)
        except json.JSONDecodeError:
            continue

        book_id = str(rec.get("book_id", "") or "").strip()
        if not book_id:
            continue

        total_books += 1

        # Chỉ xuất sách trong k-core
        if book_id not in active_items:
            continue

        # Gắn genres đã clean vào record
        rec["genres_clean"] = book_final_genres.get(book_id, [])

        fout.write(json.dumps(rec, ensure_ascii=False) + "\n")
        kept_books += 1

print(f"\n✅ Export hoàn thành.")
print(f"   Total books in file    : {total_books:,}")
print(f"   Books exported (kcore) : {kept_books:,}")
print(f"   File size              : {FILTERED_BOOKS_PATH.stat().st_size / 1024**2:.1f} MB")

# ── Verify: xem thử một vài records ──────────────────────────────────────────
print("\n── Sample records (3 đầu tiên) ─────────────────────")
with FILTERED_BOOKS_PATH.open("r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        rec = json.loads(line)
        print(f"   book_id={rec['book_id']}  |  genres_clean={rec['genres_clean']}")

---
## Phase 3 Summary — Kết quả và Files được tạo

In [ ]:
from datetime import datetime

print("=" * 65)
print("  PHASE 3 COMPLETE")
print("=" * 65)

summary = [
    ("3A — K-core filtering",  f"k={K_CORE}"),
    ("   Users giữ lại",       f"{len(active_users):,}"),
    ("   Items giữ lại",       f"{len(active_items):,}"),
    ("   Interactions",        f"{final_edges:,}"),
    ("   Density",             f"{density_after:.6f}"),
    ("3B — Genre cleaning",    f"blacklist={len(SHELF_BLACKLIST)} entries + 7 heuristic rules"),
    ("   Tags removed",        f"{len(removed_tags_log):,}"),
    ("   Unique genres kept",  f"{len(genre_kept_counter):,}"),
    ("3C — Genre transform",   f"top_k={TOP_K_GENRES}  min_doc_freq={MIN_GENRE_DOC_FREQ}"),
    ("   Vocabulary size",     f"{len(top_k_genre_set)} genres"),
    ("   Top-k genres",        ", ".join(top_k_genre_list)),
]

for label, value in summary:
    print(f"  {label:<35} {value}")

print()
print("── Output files ─────────────────────────────────────────────")
output_files = [
    (FILTERED_INTERACTIONS_PATH, "Filtered interactions (k-core)"),
    (FILTERED_BOOKS_PATH,        "Filtered books + genres_clean"),
    (OUTPUT_DIR / "removed_shelf_tags_log.csv", "Removed tags log (để review)"),
    (OUTPUT_DIR / "kcore_iteration_log.csv",    "K-core iteration log"),
]
for path, desc in output_files:
    size = f"{path.stat().st_size / 1024**2:.1f} MB" if path.exists() else "not found"
    print(f"  📄 {desc}")
    print(f"     {path.name}  [{size}]")

print()
print(f"  Generated at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 65)
print()
print("➡️  NEXT: Phase 4 — Feature Engineering + LightFM")
print("   Input files:")
print(f"   - {FILTERED_INTERACTIONS_PATH.name}")
print(f"   - {FILTERED_BOOKS_PATH.name}")